Based on LeRobot training script

In [ ]:
!rm -rf drag_rectangle_multi_task.zip drag_rectangle_multi_task

In [3]:
!pip install transformers tensorboard lerobot --ignore-installed

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 263.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 190.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 597.7/597.7 kB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 119.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.7/29.7 MB 88.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 243.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 155.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 148.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!apt update && apt install ffmpeg -y

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]                
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1939 kB]
Get:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]3m
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]m
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1271 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [33.2 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3253 kB]m
Get:11 http://archive.ubuntu.com/ubuntu jammy/main amd64 Packages [1792 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-securi

In [5]:
!apt-get install unzip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  zip
The following NEW packages will be installed:
  unzip
0 upgraded, 1 newly installed, 0 to remove and 119 not upgraded.
Need to get 175 kB of archives.
After this operation, 386 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 unzip amd64 6.0-26ubuntu3.2 [175 kB]
Fetched 175 kB in 0s (389 kB/s)
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package unzip.
(Reading database ... 26050 files and directories currently installed.)
Preparing to unpack .../unzip_6.0-26ubuntu3.2_amd64.deb ...
Unpacking unzip (6.0-26ubuntu3.2) ...
Setting up unzip (6.0-26ubuntu3.2) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...


In [6]:
!unzip drag_rectangle_multi_task.zip

Archive:  drag_rectangle_multi_task.zip
   creating: drag_rectangle_multi_task/
   creating: drag_rectangle_multi_task/images/
   creating: drag_rectangle_multi_task/images/observation.image.screen/
   creating: drag_rectangle_multi_task/meta/
  inflating: drag_rectangle_multi_task/meta/info.json  
  inflating: drag_rectangle_multi_task/meta/episodes.jsonl  
  inflating: drag_rectangle_multi_task/meta/episodes_stats.jsonl  
  inflating: drag_rectangle_multi_task/meta/tasks.jsonl  
   creating: drag_rectangle_multi_task/data/
   creating: drag_rectangle_multi_task/data/chunk-000/
  inflating: drag_rectangle_multi_task/data/chunk-000/episode_000160.parquet  
  inflating: drag_rectangle_multi_task/data/chunk-000/episode_000128.parquet  
  inflating: drag_rectangle_multi_task/data/chunk-000/episode_000181.parquet  
  inflating: drag_rectangle_multi_task/data/chunk-000/episode_000163.parquet  
  inflating: drag_rectangle_multi_task/data/chunk-000/episode_000129.parquet  
  inflating: drag_r

In [1]:
from pathlib import Path


from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.configs.types import FeatureType
from transformers import get_linear_schedule_with_warmup
from torch.utils.tensorboard import SummaryWriter  # Import TensorBoard
import os
from torchvision.transforms import v2 as transforms

import time
from datetime import timedelta


In [2]:
repo_id = "cagatayodabasi/drag_rectangle_multi_task"
repo_root = "drag_rectangle_multi_task"

In [3]:
import torch
torch.cuda.empty_cache()

In [ ]:


def main():
    # Create a directory to store the training checkpoint.
    output_directory = Path("outputs/train/newest/" + repo_id)
    output_directory.mkdir(parents=True, exist_ok=True)
    output_directory2 = Path("outputs/train/best/" + repo_id)
    output_directory2.mkdir(parents=True, exist_ok=True)


    # # Select your device
    device = torch.device("cuda")

    # Number of offline training steps (we'll only do offline training for this example.)
    # Adjust as you prefer. 5000 steps are needed to get something worth evaluating.
    training_steps = 250000
    log_freq = 250
    warmup_steps = 1000 
    image_width = 448
    image_height = 448

    # When starting from scratch (i.e. not from a pretrained policy), we need to specify 2 things before
    # creating the policy:
    #   - input/output shapes: to properly size the policy
    #   - dataset stats: for normalization and denormalization of input/outputs
    dataset_metadata = LeRobotDatasetMetadata(repo_id, root=repo_root)
    features = dataset_to_policy_features(dataset_metadata.features)
    output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}

    desired_key_inputs = {
        "observation.image.screen", 
        "observation.state"}

    input_features = {key: ft for key, ft in features.items() if key in desired_key_inputs}

    input_features['observation.image.screen'].shape = (3, image_height, image_width)

    cfg = ACTConfig(input_features=input_features, output_features=output_features)

    policy = ACTPolicy(cfg, dataset_stats=dataset_metadata.stats)
    policy.train()
    policy.to(device)

   
    delta_timestamps = {
        "action": [i / dataset_metadata.fps for i in cfg.action_delta_indices],
    }

    print(f'{cfg.action_delta_indices=}')


    augmentation_transforms = transforms.Compose([
        transforms.Resize((image_width, image_height)), 
        transforms.ColorJitter(brightness=0.2,                     # Random brightness
                           contrast=0.2,                       # Random contrast
                           saturation=0.2,                     # Random saturation
                           hue=0.1),                           # Random hue
    ])

    dataset = LeRobotDataset(repo_id, root=repo_root ,image_transforms=augmentation_transforms, delta_timestamps=delta_timestamps)


    optimizer = torch.optim.Adam(policy.parameters(), lr=1e-4)
    dataloader = torch.utils.data.DataLoader(
        dataset,
        num_workers=32,
        batch_size=64,
        shuffle=True,
        pin_memory=device.type != "cpu",
        drop_last=True,
    )

    print("dataloader")

    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=training_steps)

    print("scheduler")
    # Run training loop.
    step = 0
    done = False
    best_loss = float('inf')  # Initialize the best loss as infinity

    start_time = time.time()  # Time tracking begins

    while not done:
        for batch in dataloader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != 'task'}
            loss, _ = policy.forward(batch)
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            if step % log_freq == 0:
                elapsed_time = time.time() - start_time
                avg_time_per_step = elapsed_time / (step + 1)
                steps_remaining = training_steps - step
                eta_seconds = avg_time_per_step * steps_remaining
                eta = str(timedelta(seconds=int(eta_seconds)))

                print(f"step: {step} | loss: {loss.item():.3f} | lr: {scheduler.get_last_lr()[0]:.6f} | ETA: {eta}")
                
                policy.save_pretrained(output_directory)

                if loss.item() < best_loss:
                    best_loss = loss.item()
                    print(f"New best loss: {best_loss:.3f}. Saving model.")
                    best_model_directory = os.path.join(output_directory2, f"best_model_loss_{best_loss:.3f}")
                    policy.save_pretrained(best_model_directory)

            step += 1
            if step >= training_steps:
                done = True
                break


 

if __name__ == "__main__":
    main()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 41.9MB/s]


cfg.action_delta_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


Resolving data files:   0%|          | 0/201 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

dataloader
scheduler
step: 0 | loss: 72.440 | lr: 0.000000 | ETA: 110 days, 23:46:52
New best loss: 72.440. Saving model.
step: 250 | loss: 2.524 | lr: 0.000025 | ETA: 2 days, 13:29:24
New best loss: 2.524. Saving model.
step: 500 | loss: 1.622 | lr: 0.000050 | ETA: 2 days, 15:26:14
New best loss: 1.622. Saving model.
step: 750 | loss: 0.955 | lr: 0.000075 | ETA: 2 days, 15:03:15
New best loss: 0.955. Saving model.
step: 1000 | loss: 0.562 | lr: 0.000100 | ETA: 2 days, 15:57:04
New best loss: 0.562. Saving model.
step: 1250 | loss: 0.316 | lr: 0.000100 | ETA: 2 days, 20:37:14
New best loss: 0.316. Saving model.
step: 1500 | loss: 0.206 | lr: 0.000100 | ETA: 2 days, 18:26:59
New best loss: 0.206. Saving model.
step: 1750 | loss: 0.177 | lr: 0.000100 | ETA: 2 days, 16:39:11
New best loss: 0.177. Saving model.
step: 2000 | loss: 0.137 | lr: 0.000100 | ETA: 2 days, 14:28:46
New best loss: 0.137. Saving model.
step: 2250 | loss: 0.119 | lr: 0.000099 | ETA: 2 days, 13:47:11
New best loss: 0.